# Hodge Curl Citation Cartel Detector

This notebook demonstrates the **Hodge-Curl Cartel Detector** for identifying citation manipulation in academic journal networks.

**Core idea:** The net-flow citation matrix between journals is decomposed via combinatorial Hodge decomposition into three orthogonal components:
- **Gradient** (Y_grad): flow explained by a global prestige ranking (HodgeRank)
- **Curl** (Y_curl): cyclic flow *not* explainable by any ranking — the fingerprint of citation cartels
- **Harmonic** (Y_harm): non-cyclic, non-gradient residual flow

Journals with high curl scores form cyclic citation rings that cannot be explained by legitimate prestige hierarchies.

**Pipeline:**
1. Generate (or load) a journal citation network
2. Preprocess: threshold edges, compute net-flow matrix
3. Enumerate triangles; build incidence matrices B1, B2
4. Hodge decomposition via sparse least squares
5. Null model (row-permutation) to calibrate z-scores
6. Baselines: CIDRE, reciprocity, PageRank, within-group density
7. Evaluation against ground truth suppressed journals
8. Synthetic cartel injection sensitivity test
9. Confound test (curl vs density/reciprocity)


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru and aiohttp are NOT pre-installed on Colab
_pip('loguru==0.7.3')
_pip('aiohttp==3.11.18')

# Core packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1',
         'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
import sys
import os
import gc
import json
import math
import time
import resource
import multiprocessing as mp
from pathlib import Path
from collections import defaultdict
from typing import Optional, Dict, List, Tuple, Any
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import scipy.sparse as sparse
from scipy.sparse.linalg import lsqr
from loguru import logger
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import matplotlib

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Notebook workspace (replaces __file__-based WORKSPACE from method.py)
WORKSPACE = Path('.').resolve()
(WORKSPACE / 'logs').mkdir(exist_ok=True)
(WORKSPACE / 'data').mkdir(exist_ok=True)
(WORKSPACE / 'results').mkdir(exist_ok=True)

print('Imports OK')

## Load Pre-computed Results

We load `mini_demo_data.json` — a curated subset of 100 journal records with all Hodge scores already computed — to illustrate the output schema and use for the final visualization.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d509fc-curl-in-the-citation-graph-hodge-flow-de/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data['datasets'][0]['examples']
print(f"Loaded {len(examples)} examples")
from collections import Counter
print("Label distribution:", Counter(e['output'] for e in examples))
print("\nSample record:")
print(json.dumps(examples[0], indent=2)[:600], '...')

## Configuration

All tunable parameters are defined here. Values are set to a **minimal demo scale** that runs in seconds on CPU. The original paper values are shown in comments.

In [ ]:
# --- Synthetic network ---
N_JOURNALS   = 150    # journals in synthetic network  (original: 800)
N_FIELDS     = 5      # subject fields                 (original: 12)
N_CARTELS    = 4      # injected cartel rings          (original: 10)
CARTEL_SIZE  = 3      # nodes per cartel (3 = triangle)(original: 5)
SEED         = 42

# --- Preprocessing ---
THRESH       = 3      # min bidirectional edge weight  (original: 3)

# --- Null model ---
N_NULL       = 10     # row-permutation null samples   (original: 100)

# --- Evaluation bootstrap ---
B_BOOTSTRAP  = 200    # bootstrap resamples for CI     (original: 2000)

# --- Synthetic injection sensitivity test ---
K_VALUES     = [3]          # cartel sizes to test    (original: [3,5,10,20])
W_FACTORS    = [0.1, 1.0]   # strength multipliers    (original: [0.01,0.05,0.1,0.3,0.5,1.0,2.0])
N_REPEATS    = 3     # repeats per condition           (original: 20)

# --- Confound test bootstrap ---
CONFOUND_BOOT = 200   # bootstrap samples              (original: 2000)

print("Config set — demo scale")

## Phase 0: Ground Truth

In the real experiment, ground truth comes from public records of JCR-suppressed journals (CIDRE paper, Retraction Watch). Here we use the synthetic network's injected cartel nodes as ground truth.

In [ ]:
# ============================================================
# PHASE 0: GROUND TRUTH COMPILATION
# ============================================================
# Known JCR-suppressed journals from public records (CIDRE paper, Retractionwatch)
GROUND_TRUTH_HARDCODED: List[Dict] = [
    # ---- Citation stacking (primary evaluation target) ----
    # Romanian/Eastern European ring
    {"name": "Romanian Journal of Legal Medicine", "issn_l": "1222-5975", "reason": "citation_stacking"},
    {"name": "Materiale Plastice", "issn_l": "0025-5289", "reason": "citation_stacking"},
    {"name": "REVISTA DE CHIMIE", "issn_l": "0034-7752", "reason": "citation_stacking"},
    {"name": "Industria Textila", "issn_l": "1222-5347", "reason": "citation_stacking"},
    {"name": "Journal of Environmental Protection and Ecology", "issn_l": "1311-5065", "reason": "citation_stacking"},
    {"name": "Textile and Leather Review", "issn_l": "2623-6346", "reason": "citation_stacking"},
    # Turkish ring
    {"name": "Ekoloji", "issn_l": "1300-1361", "reason": "citation_stacking"},
    {"name": "Journal of International Environmental Application and Science", "issn_l": "1307-0428", "reason": "citation_stacking"},
    # Pakistani ring
    {"name": "Pakistan Journal of Zoology", "issn_l": "0030-9923", "reason": "citation_stacking"},
    {"name": "Journal of Animal and Plant Sciences", "issn_l": "1018-7081", "reason": "citation_stacking"},
    # Chemistry/materials ring
    {"name": "Fresenius Environmental Bulletin", "issn_l": "1018-4619", "reason": "citation_stacking"},
    {"name": "Cellular and Molecular Biology", "issn_l": "0145-5680", "reason": "citation_stacking"},
    # ---- Self-citation (secondary evaluation) ----
    {"name": "Tumor Biology", "issn_l": "1010-4283", "reason": "self_citation"},
]


def build_ground_truth() -> List[Dict]:
    """Return ground truth list with OpenAlex IDs resolved."""
    gt = [dict(g) for g in GROUND_TRUTH_HARDCODED]
    logger.info(f"Ground truth: {len(gt)} suppressed journals "
                f"({sum(1 for g in gt if g['reason']=='citation_stacking')} stacking, "
                f"{sum(1 for g in gt if g['reason']=='self_citation')} self-citation)")
    return gt


ground_truth = build_ground_truth()
gt_issns = {g['issn_l'] for g in ground_truth}

## Phase 1: Synthetic Citation Network

We generate a realistic synthetic citation network (since OpenAlex API requires rate limits for production use). The generator:
- Creates N journals in F fields with prestige-weighted within-field citations
- Injects `n_cartels` directed rings (A→B→C→A) with high-weight cyclic flows
- These rings have non-zero Hodge curl, while legitimate prestige flows have near-zero curl

In [ ]:
# ============================================================
# SYNTHETIC NETWORK (fallback if API unavailable)
# ============================================================
def generate_synthetic_network(N: int = 500, n_fields: int = 10,
                                 n_cartels: int = 8, cartel_size: int = 5,
                                 seed: int = 42) -> Tuple[sparse.csr_matrix, List[Dict], List[Dict]]:
    """Generate a realistic synthetic citation network with injected cartels."""
    rng = np.random.RandomState(seed)
    logger.info(f"Generating synthetic network: N={N}, fields={n_fields}, cartels={n_cartels}")

    field_labels = np.repeat(np.arange(n_fields), N // n_fields + 1)[:N]

    # Prestige scores: roughly hierarchical
    prestige = rng.exponential(scale=1.0, size=N)

    # Base citations: prestige-based (higher prestige gets cited more)
    C = sparse.lil_matrix((N, N), dtype=np.float64)

    # Within-field citations (dense)
    for field in range(n_fields):
        members = np.where(field_labels == field)[0]
        for i in members:
            # Each journal cites ~20 others in same field, proportional to prestige
            targets = rng.choice(members, size=min(20, len(members) - 1), replace=False)
            weights = prestige[targets] / (prestige[targets].sum() + 1e-10)
            for t, w in zip(targets, weights):
                if t != i:
                    C[i, t] += max(1, int(rng.poisson(50 * w * prestige[i])))

    # Cross-field citations (sparse)
    for i in range(N):
        n_cross = rng.poisson(3)
        if n_cross > 0:
            targets = rng.choice(N, size=n_cross, replace=False)
            for t in targets:
                if t != i:
                    C[i, t] += max(1, int(rng.poisson(5 * prestige[t])))

    # Inject cartels as 3-node directed rings (triangles) — detectable by Hodge curl.
    # Citation stacking ring: A→B, B→C, C→A at high rate; small reverse to mimic legitimacy.
    # Using k=3 guarantees triangle formation (required for non-zero triangle curl).
    cartel_nodes_all = []
    all_nodes = list(range(N))
    available = set(all_nodes)
    all_vals_init = [v for row in C.data for v in row]
    w_cartel = int(max(all_vals_init) * 0.6) if all_vals_init else 100

    for c in range(n_cartels):
        k_use = min(cartel_size, 3)  # 3-node triangle for guaranteed curl detectability
        if len(available) < k_use:
            break
        nodes = sorted(rng.choice(list(available), size=k_use, replace=False))
        cartel_nodes_all.extend(nodes)
        available -= set(nodes)

        # Directed ring A→B→C→A (net cyclic flow = high curl in triangle)
        for idx in range(k_use):
            u, v = nodes[idx], nodes[(idx + 1) % k_use]
            C[u, v] += w_cartel          # strong directed citation
            C[v, u] += w_cartel * 0.15  # small reverse (realistic noise)

    C_csr = C.tocsr()

    # Build journal list
    field_names = ["Biology", "Chemistry", "Physics", "Medicine", "Engineering",
                   "Mathematics", "Computer Science", "Environmental Science",
                   "Agriculture", "Materials Science"]
    journals = []
    for i in range(N):
        j_name = f"Journal_{i:04d}_{field_names[field_labels[i] % len(field_names)]}"
        journals.append({
            "id": f"https://openalex.org/S{i+1000000:08d}",
            "display_name": j_name,
            "issn_l": f"{1000+i//100:04d}-{i%1000:04d}",
            "cited_by_count": int(C_csr[:, i].sum()),
            "idx": i,
            "synthetic_field": field_names[field_labels[i] % len(field_names)],
        })

    # Build ground truth: cartel nodes are "suppressed for citation_stacking"
    gt = []
    for node in cartel_nodes_all:
        gt.append({
            "name": journals[node]["display_name"],
            "issn_l": journals[node]["issn_l"],
            "reason": "citation_stacking",
            "openalex_id": journals[node]["id"],
        })

    logger.info(f"Synthetic network: {N} nodes, {C_csr.nnz} edges, {len(cartel_nodes_all)} cartel nodes")
    return C_csr, journals, gt


C_raw, journals_raw, ground_truth = generate_synthetic_network(
    N=N_JOURNALS, n_fields=N_FIELDS, n_cartels=N_CARTELS, cartel_size=CARTEL_SIZE, seed=SEED
)
gt_issns = {g['issn_l'] for g in ground_truth}
is_synthetic = True

## Phase 2: Preprocessing

We threshold edges (keep only pairs where total bidirectional flow ≥ THRESH), remove isolated nodes, and compute the **net-flow matrix** Y = C − Cᵀ.

The net-flow Y is the key object: Y[i,j] > 0 means journal i cites journal j more than vice versa. This anti-symmetric matrix is then decomposed by Hodge theory.

In [ ]:
# ============================================================
# PHASE 2: PREPROCESSING
# ============================================================
def preprocess(C: sparse.csr_matrix, thresh: int = THRESH) -> Dict:
    """Threshold, remove isolates, build net-flow and edge list."""
    N_raw = C.shape[0]

    # Edge threshold
    C_plus_CT = C + C.T
    mask = C_plus_CT >= thresh
    C_thresh = C.multiply(mask)
    C_thresh.eliminate_zeros()

    # Remove isolated nodes
    row_sums = np.asarray(C_thresh.sum(1)).squeeze()
    col_sums = np.asarray(C_thresh.sum(0)).squeeze()
    active_mask = (row_sums + col_sums) > 0
    active_nodes = np.where(active_mask)[0]
    N = len(active_nodes)

    C_active = C_thresh[active_nodes][:, active_nodes].tocsr()

    # Net flow matrix
    Y = C_active - C_active.T

    # Oriented edge list (canonical: i < j)
    cx = C_active.tocoo()
    edges_set = set()
    for i, j in zip(cx.row, cx.col):
        if i != j:
            edges_set.add((min(int(i), int(j)), max(int(i), int(j))))
    edges = sorted(edges_set)
    E = len(edges)
    edge_to_idx = {e: idx for idx, e in enumerate(edges)}

    # Edge flow vector
    Y_arr = np.asarray(Y.todense())
    Y_e = np.array([Y_arr[i, j] for (i, j) in edges], dtype=np.float64)

    logger.info(f"Preprocessing: N_raw={N_raw} → N_active={N}, E={E}, "
                f"density={2*E/(N*(N-1)+1e-10):.4f}")
    del C_thresh, C_plus_CT, mask, cx
    gc.collect()

    return {
        "C_active": C_active,
        "active_nodes": active_nodes,
        "Y_e": Y_e,
        "edges": edges,
        "edge_to_idx": edge_to_idx,
        "N": N,
        "E": E,
    }


prep = preprocess(C_raw, thresh=THRESH)
C_active = prep["C_active"]
active_nodes = prep["active_nodes"]
edges = prep["edges"]
edge_to_idx = prep["edge_to_idx"]
Y_e = prep["Y_e"]
N = prep["N"]
E = prep["E"]

del C_raw
gc.collect()

## Phase 3: Hodge Decomposition

The Hodge decomposition Y_e = Y_grad + Y_curl + Y_harm is computed via:
1. **B1** (N×E): node-edge incidence matrix (tail=-1, head=+1)
2. **B2** (E×T): edge-triangle incidence matrix (encodes triangle orientations)

The gradient component is the least-squares solution to B1ᵀs ≈ Y_e (HodgeRank). The residual is then projected onto the image of B2 to get the curl, with the remainder being harmonic.

Key metric: **node_grad_residual** = average |Y_e − Y_grad| on a node's incident edges. High residual means the journal's citation flows cannot be explained by any prestige ordering — the signature of cyclic manipulation.

In [ ]:
# ============================================================
# PHASE 3: HODGE DECOMPOSITION
# ============================================================
def enumerate_triangles(edges: List[Tuple[int, int]], N: int,
                         edge_to_idx: Dict) -> List[Tuple[int, int, int]]:
    """Enumerate all triangles (3-cliques) in the undirected graph."""
    adj_list = defaultdict(set)
    for (i, j) in edges:
        adj_list[i].add(j)
        adj_list[j].add(i)

    triangles = []
    for (i, j) in edges:
        common = adj_list[i] & adj_list[j]
        for k in common:
            if k > j:
                triangles.append((i, j, k))

    logger.info(f"Triangle enumeration: {len(triangles):,} triangles")
    return triangles


def build_incidence_matrices(N: int, E: int, edges: List[Tuple[int, int]],
                               edge_to_idx: Dict, triangles: List[Tuple[int, int, int]],
                               use_direct: bool = False) -> Dict:
    """Build B1 (node×edge) and B2 (edge×triangle) incidence matrices."""
    # B1: node×edge
    rows_B1, cols_B1, data_B1 = [], [], []
    for e_idx, (i, j) in enumerate(edges):
        rows_B1.extend([i, j])
        cols_B1.extend([e_idx, e_idx])
        data_B1.extend([-1.0, 1.0])  # tail=-1, head=+1
    B1 = sparse.csr_matrix((data_B1, (rows_B1, cols_B1)), shape=(N, E))

    if use_direct or not triangles:
        return {"B1": B1, "B2": None, "use_direct": True}

    T = len(triangles)
    # B2: edge×triangle
    # Convention: triangle (i,j,k) i<j<k, circuit i→j→k→i
    # e_ij: +1 (i→j matches), e_jk: +1 (j→k matches), e_ik: -1 (circuit goes k→i, reversed)
    rows_B2, cols_B2, data_B2 = [], [], []
    for t_idx, (i, j, k) in enumerate(triangles):
        e_ij = edge_to_idx.get((i, j))
        e_jk = edge_to_idx.get((j, k))
        e_ik = edge_to_idx.get((i, k))
        if e_ij is None or e_jk is None or e_ik is None:
            continue
        rows_B2.extend([e_ij, e_jk, e_ik])
        cols_B2.extend([t_idx, t_idx, t_idx])
        data_B2.extend([1.0, 1.0, -1.0])
    B2 = sparse.csr_matrix((data_B2, (rows_B2, cols_B2)), shape=(E, T))

    # Verify Hodge identity: B1 @ B2 ≈ 0
    check = B1 @ B2
    max_err = abs(check.data).max() if len(check.data) > 0 else 0.0
    if max_err > 1e-8:
        logger.warning(f"Hodge identity violation: max_err={max_err:.2e}")
    else:
        logger.info(f"Hodge identity verified (max_err={max_err:.2e})")

    return {"B1": B1, "B2": B2, "use_direct": False}


triangles = enumerate_triangles(edges, N, edge_to_idx)
T = len(triangles)
MAX_TRIANGLES = 3_000_000  # use Fallback D if exceeded
use_direct = T > MAX_TRIANGLES
if use_direct:
    logger.warning(f"Too many triangles ({T:,} > {MAX_TRIANGLES:,}), using Fallback D")

inc = build_incidence_matrices(N, E, edges, edge_to_idx, triangles, use_direct=use_direct)
B1 = inc["B1"]
B2 = inc["B2"]

In [ ]:
def hodge_decompose(Y_e: np.ndarray, B1: sparse.csr_matrix,
                     B2: Optional[sparse.csr_matrix], edges: List[Tuple[int, int]],
                     triangles: List[Tuple[int, int, int]], N: int,
                     use_direct: bool = False) -> Dict:
    """
    Full Hodge decomposition: Y_e = Y_grad + Y_curl + Y_harm.
    Returns prestige scores, curl components, and energy fractions.
    """
    E = len(Y_e)

    # Gradient component: solve min_s ||B1^T @ s - Y_e||^2
    logger.info("Solving gradient (HodgeRank prestige)...")
    result_grad = lsqr(B1.T, Y_e, damp=1e-6, atol=1e-10, btol=1e-10, iter_lim=20000)
    s_star = result_grad[0]  # N-vector: prestige scores
    Y_grad = B1.T @ s_star
    residual = Y_e - Y_grad

    logger.info(f"Gradient solved: exit_code={result_grad[1]}, resid_norm={result_grad[4]:.4f}")

    # Per-node gradient residual score: detects any cycle length (not just triangles)
    node_res_sum = np.zeros(N)
    node_edge_count_res = np.zeros(N, dtype=int)
    for e_idx, (i, j) in enumerate(edges):
        res_val = abs(residual[e_idx])
        node_res_sum[i] += res_val
        node_res_sum[j] += res_val
        node_edge_count_res[i] += 1
        node_edge_count_res[j] += 1
    node_grad_residual = node_res_sum / (node_edge_count_res + 1e-10)

    # Curl component
    if not use_direct and B2 is not None:
        logger.info("Solving curl (Hodge curl component)...")
        # B2 is E×T; solve min_h ||B2 @ h - residual||^2 for T-vector h
        result_curl = lsqr(B2, residual, damp=1e-6, atol=1e-10, btol=1e-10, iter_lim=20000)
        h_star = result_curl[0]
        Y_curl_vec = B2 @ h_star
        triangle_curls = B2.T @ Y_e  # per-triangle curl amplitude (T-vector)
        logger.info(f"Curl solved: exit_code={result_curl[1]}")
    else:
        # Fallback D: direct triangle aggregation
        logger.info("Using direct triangle curl (Fallback D)...")
        if triangles:
            Y_arr = np.zeros((N, N))
            for e_idx, (i, j) in enumerate(edges):
                Y_arr[i, j] = Y_e[e_idx]
                Y_arr[j, i] = -Y_e[e_idx]
            triangle_curls = np.array([
                Y_arr[i, j] + Y_arr[j, k] - Y_arr[i, k]
                for (i, j, k) in triangles
            ])
            # Project onto edge space for energy calculation
            T = len(triangles)
            rows, cols, data = [], [], []
            edge_to_idx_local = {e: eid for eid, e in enumerate(edges)}
            for t_idx, (i, j, k) in enumerate(triangles):
                for (ei, ej), sign in [((i,j),1), ((j,k),1), ((i,k),-1)]:
                    eid = edge_to_idx_local.get((ei, ej))
                    if eid is not None:
                        rows.append(eid); cols.append(t_idx); data.append(float(sign))
            B2_approx = sparse.csr_matrix((data, (rows, cols)), shape=(E, T))
            result_curl2 = lsqr(B2_approx, residual, damp=1e-6, iter_lim=5000)
            Y_curl_vec = B2_approx @ result_curl2[0]
            del B2_approx, Y_arr
            gc.collect()
        else:
            triangle_curls = np.array([])
            Y_curl_vec = np.zeros(E)

    Y_harm = residual - Y_curl_vec

    # Energy fractions
    total_energy = np.dot(Y_e, Y_e)
    if total_energy < 1e-15:
        grad_frac, curl_frac, harm_frac = 0.0, 0.0, 0.0
    else:
        grad_frac = float(np.dot(Y_grad, Y_grad) / total_energy)
        curl_frac = float(np.dot(Y_curl_vec, Y_curl_vec) / total_energy)
        harm_frac = float(np.dot(Y_harm, Y_harm) / total_energy)
    logger.info(f"Energy: grad={grad_frac:.3f}, curl={curl_frac:.3f}, harm={harm_frac:.3f} "
                f"(sum={grad_frac+curl_frac+harm_frac:.3f})")

    # Per-node curl scores from triangles
    node_curl_sum = np.zeros(N)
    node_tri_count = np.zeros(N, dtype=int)
    if len(triangle_curls) > 0:
        for t_idx, (i, j, k) in enumerate(triangles):
            val = abs(triangle_curls[t_idx])
            for node in [i, j, k]:
                node_curl_sum[node] += val
                node_tri_count[node] += 1
    node_curl_score = node_curl_sum / (node_tri_count + 1e-10)

    return {
        "s_star": s_star,
        "Y_grad": Y_grad,
        "Y_curl": Y_curl_vec,
        "Y_harm": Y_harm,
        "triangle_curls": triangle_curls,
        "node_curl_score": node_curl_score,
        "node_tri_count": node_tri_count,
        "node_grad_residual": node_grad_residual,
        "grad_fraction": grad_frac,
        "curl_fraction": curl_frac,
        "harm_fraction": harm_frac,
    }


def degree_normalize_curl(node_curl_score: np.ndarray, C_active: sparse.csr_matrix) -> np.ndarray:
    """Degree-normalize curl score to reduce high-degree bias."""
    degree = np.asarray(C_active.sum(1)).squeeze() + np.asarray(C_active.sum(0)).squeeze()
    return node_curl_score / (np.log1p(degree) + 1e-10)


hodge = hodge_decompose(Y_e, B1, B2, edges, triangles, N, use_direct=use_direct)

## Phase 4: Null Model

A **degree-preserving row-permutation null** calibrates the curl scores into z-scores. For each of N_NULL samples, we randomly permute which journals receive citations from each citing journal (preserving the volume of citations from each source). The resulting null distribution of curl scores tells us how high a score must be to be statistically anomalous.

In [ ]:
# ============================================================
# PHASE 4: NULL MODEL (multiprocessing)
# ============================================================
def _null_worker(args: Tuple) -> np.ndarray:
    """Worker for one null model sample. Returns node_curl_scores."""
    (C_data, C_indices, C_indptr, shape, edges_arr, triangles_arr, seed) = args
    rng = np.random.RandomState(seed)
    N, _ = shape
    E = len(edges_arr)

    # Reconstruct C
    C = sparse.csr_matrix((C_data.copy(), C_indices.copy(), C_indptr.copy()), shape=shape)
    # Row-permutation null: shuffle which journals receive citations from each row
    C_lil = C.tolil()
    for i in range(N):
        if len(C_lil.data[i]) > 1:
            perm = rng.permutation(len(C_lil.data[i]))
            C_lil.data[i] = [C_lil.data[i][p] for p in perm]
    C_null = C_lil.tocsr()
    Y_null = C_null - C_null.T
    Y_arr = Y_null.toarray()

    # Build Y_e for null
    Y_e_null = np.array([Y_arr[edges_arr[k, 0], edges_arr[k, 1]] for k in range(E)])

    # Direct triangle curl (fast for null model)
    node_curl = np.zeros(N)
    if len(triangles_arr) > 0:
        for t in range(len(triangles_arr)):
            i, j, k = triangles_arr[t]
            tc = abs(Y_arr[i, j] + Y_arr[j, k] - Y_arr[i, k])
            node_curl[i] += tc
            node_curl[j] += tc
            node_curl[k] += tc
        tri_count = np.zeros(N, dtype=int)
        for t in range(len(triangles_arr)):
            for node in triangles_arr[t]:
                tri_count[node] += 1
        node_curl = node_curl / (tri_count + 1e-10)

    return node_curl


def compute_null_model(C_active: sparse.csr_matrix, edges: List[Tuple[int, int]],
                        triangles: List[Tuple[int, int, int]], N: int,
                        n_samples: int = N_NULL) -> Dict:
    """Compute null model calibration via row-permutation null."""
    logger.info(f"Computing null model: {n_samples} samples...")

    edges_arr = np.array(edges, dtype=np.int32) if edges else np.zeros((0, 2), dtype=np.int32)
    triangles_arr = np.array(triangles, dtype=np.int32) if triangles else np.zeros((0, 3), dtype=np.int32)

    C_csr = C_active.tocsr()
    worker_args = [
        (C_csr.data, C_csr.indices, C_csr.indptr, C_csr.shape,
         edges_arr, triangles_arr, seed)
        for seed in range(n_samples)
    ]

    null_scores = []
    n_workers = max(1, min(4, mp.cpu_count() - 1))

    with ProcessPoolExecutor(max_workers=n_workers,
                              mp_context=mp.get_context("spawn")) as pool:
        futures = {pool.submit(_null_worker, args): i for i, args in enumerate(worker_args)}
        done = 0
        for future in as_completed(futures):
            try:
                result = future.result()
                null_scores.append(result)
                done += 1
            except Exception as e:
                logger.error(f"Null worker failed: {e}")

    if not null_scores:
        logger.error("No null samples computed, using fallback")
        return {"z_score": np.zeros(N), "p_value": np.ones(N),
                "null_mean": np.zeros(N), "null_std": np.ones(N)}

    null_matrix = np.stack(null_scores, axis=0)  # (n_samples, N)
    null_mean = null_matrix.mean(0)
    null_std = null_matrix.std(0) + 1e-10

    logger.info(f"Null model complete: {len(null_scores)} samples used")
    return {
        "null_mean": null_mean,
        "null_std": null_std,
        "null_matrix": null_matrix,
    }


def compute_z_scores(node_curl_score: np.ndarray, null_stats: Dict, N: int) -> Dict:
    """Compute per-node z-scores and p-values relative to null model."""
    null_mean = null_stats["null_mean"]
    null_std = null_stats["null_std"]
    null_matrix = null_stats.get("null_matrix")

    z_score = (node_curl_score - null_mean) / null_std

    if null_matrix is not None:
        p_value = (null_matrix >= node_curl_score[np.newaxis, :]).mean(0)
    else:
        p_value = np.ones(N)

    return {"z_score": z_score, "p_value": p_value}


null_stats = compute_null_model(C_active, edges, triangles, N, n_samples=N_NULL)
z_info = compute_z_scores(hodge["node_curl_score"], null_stats, N)

# Clean up large matrix
del null_stats["null_matrix"]
gc.collect()

## Phase 5: Baselines

We compute four baselines for comparison:
- **CIDRE** (approx): Poisson null within Louvain communities — the state-of-the-art citation cartel detector
- **Reciprocity**: per-node weighted fraction of symmetric citations (high reciprocity ≈ mutual citation trading)
- **Within-group density**: how densely a node's community cites itself (proxy for clique structure)
- **PageRank**: journal prestige — high-prestige journals should *not* be flagged

In [ ]:
# ============================================================
# PHASE 5: BASELINES
# ============================================================
def compute_reciprocity(C_active: sparse.csr_matrix) -> np.ndarray:
    """Per-node weighted reciprocity score."""
    N = C_active.shape[0]
    recip = np.zeros(N)
    C_arr = C_active.toarray()

    for i in range(N):
        row = C_arr[i]
        partners = np.where(row > 0)[0]
        if len(partners) == 0:
            continue
        vals = []
        weights = []
        for j in partners:
            c_ij = C_arr[i, j]
            c_ji = C_arr[j, i]
            total = c_ij + c_ji
            if total > 0:
                vals.append(min(c_ij, c_ji) / total)
                weights.append(total)
        if weights:
            recip[i] = np.average(vals, weights=weights)

    return recip


def compute_community_density(C_active: sparse.csr_matrix, comm_labels: np.ndarray,
                               N: int) -> np.ndarray:
    """Per-node within-community citation density."""
    communities = defaultdict(list)
    for node, c in enumerate(comm_labels):
        communities[c].append(node)

    density = np.zeros(N)
    for c, members in communities.items():
        if len(members) < 2:
            continue
        sub = C_active[members][:, members]
        internal = sub.sum()
        possible = len(members) * (len(members) - 1)
        d = float(internal) / (possible + 1e-10)
        for node in members:
            density[node] = d

    return density


def compute_pagerank(C_active: sparse.csr_matrix) -> np.ndarray:
    """PageRank on directed citation graph."""
    G = nx.from_scipy_sparse_array(C_active, create_using=nx.DiGraph())
    pr = nx.pagerank(G, alpha=0.85, max_iter=200, tol=1e-6)
    return np.array([pr.get(n, 0.0) for n in range(C_active.shape[0])])


def simple_cidre_baseline(C_active: sparse.csr_matrix, comm_labels: np.ndarray) -> np.ndarray:
    """
    CIDRE-inspired baseline: Poisson null within communities.
    Score = max over community partners of observed/expected ratio.
    """
    N = C_active.shape[0]
    scores = np.zeros(N)
    communities = defaultdict(list)
    for node, c in enumerate(comm_labels):
        communities[c].append(node)

    for c, members in communities.items():
        if len(members) < 2:
            continue
        sub = C_active[members][:, members].toarray().astype(float)
        total = sub.sum()
        if total < 1:
            continue
        row_sums = sub.sum(1)
        col_sums = sub.sum(0)
        expected = np.outer(row_sums, col_sums) / (total + 1e-10)
        # Zero out diagonal
        np.fill_diagonal(expected, 0)
        np.fill_diagonal(sub, 0)
        ratio = sub / (expected + 1e-10)
        for i, node in enumerate(members):
            # Anomaly score: max outgoing excess ratio
            scores[node] = max(scores[node], float(ratio[i].max()))

    return scores


def compute_all_baselines(C_active: sparse.csr_matrix, N: int) -> Dict[str, np.ndarray]:
    """Compute all baseline scores."""
    logger.info("Computing baselines...")

    # Community detection with Louvain
    G_und = nx.from_scipy_sparse_array((C_active + C_active.T) / 2)
    try:
        communities = nx.community.louvain_communities(G_und, seed=42, weight="weight")
        comm_labels = np.zeros(N, dtype=int)
        for c_idx, comm in enumerate(communities):
            for node in comm:
                comm_labels[node] = c_idx
        n_comms = len(communities)
    except Exception as e:
        logger.warning(f"Louvain failed: {e}, using degree-based communities")
        degrees = np.asarray(C_active.sum(1)).squeeze()
        comm_labels = (degrees / (degrees.max() / 10 + 1e-10)).astype(int).clip(0, 9)
        n_comms = 10

    logger.info(f"  Louvain: {n_comms} communities")

    recip = compute_reciprocity(C_active)
    logger.info("  Reciprocity done")

    density = compute_community_density(C_active, comm_labels, N)
    logger.info("  Community density done")

    pr = compute_pagerank(C_active)
    logger.info("  PageRank done")

    cidre = simple_cidre_baseline(C_active, comm_labels)
    logger.info("  CIDRE (approx) done")

    return {
        "reciprocity": recip,
        "within_group_density": density,
        "pagerank": pr,
        "cidre": cidre,
        "comm_labels": comm_labels,
    }


baselines = compute_all_baselines(C_active, N)
comm_labels = baselines.pop("comm_labels")

## Phase 6: Evaluation

We match the suppressed/cartel journals to their active node indices and compute AUC-ROC and average precision for each detection method, with 95% bootstrap confidence intervals.

In [ ]:
# ============================================================
# PHASE 6: EVALUATION
# ============================================================
def match_ground_truth(journals: List[Dict], ground_truth: List[Dict],
                        active_nodes: np.ndarray) -> Dict:
    """Map suppressed journals to active node indices."""
    # Build lookup by ISSN
    issn_to_active_idx = {}
    for active_idx, raw_idx in enumerate(active_nodes):
        j = journals[raw_idx]
        issn = j.get("issn_l") or j.get("issn_l", "")
        if issn:
            issn_to_active_idx[issn] = active_idx
        # Also try other ISSNs
        for issn_alt in (j.get("issn") or []):
            if issn_alt and issn_alt not in issn_to_active_idx:
                issn_to_active_idx[issn_alt] = active_idx

    N_active = len(active_nodes)
    labels_stacking = np.zeros(N_active)
    labels_all = np.zeros(N_active)
    matched = []

    for gt in ground_truth:
        issn = gt.get("issn_l", "")
        active_idx = issn_to_active_idx.get(issn)
        if active_idx is not None:
            if gt["reason"] == "citation_stacking":
                labels_stacking[active_idx] = 1
            labels_all[active_idx] = 1
            matched.append({**gt, "active_idx": active_idx})

    n_stacking = int(labels_stacking.sum())
    n_all = int(labels_all.sum())
    logger.info(f"Ground truth matched: {n_stacking} stacking, {n_all} total "
                f"(out of {len(ground_truth)} suppressed)")

    return {
        "labels_stacking": labels_stacking,
        "labels_all": labels_all,
        "matched": matched,
        "n_stacking": n_stacking,
        "n_all": n_all,
    }


def evaluate_method(scores: np.ndarray, labels: np.ndarray,
                     method_name: str, B: int = B_BOOTSTRAP) -> Dict:
    """Compute AUC, AP, Precision@k with bootstrap CI."""
    if labels.sum() < 2:
        logger.warning(f"Too few positives ({labels.sum()}) for {method_name}")
        return {"auc": None, "auc_pr": None, "prec_at_k": {}, "ci": [None, None]}

    # Handle NaN/Inf
    scores_clean = np.where(np.isfinite(scores), scores, 0.0)

    try:
        auc = float(roc_auc_score(labels, scores_clean))
        auc_pr = float(average_precision_score(labels, scores_clean))
    except Exception as e:
        logger.error(f"AUC error for {method_name}: {e}")
        return {"auc": None, "auc_pr": None, "prec_at_k": {}, "ci": [None, None]}

    ranked = np.argsort(scores_clean)[::-1]
    prec_at_k = {}
    for k in [10, 50, 100]:
        if k <= len(labels):
            prec_at_k[str(k)] = float(labels[ranked[:k]].mean())

    # Bootstrap CI
    boot_aucs = []
    rng = np.random.RandomState(42)
    for _ in range(B):
        idx = rng.randint(0, len(labels), len(labels))
        if labels[idx].sum() > 0:
            try:
                boot_aucs.append(roc_auc_score(labels[idx], scores_clean[idx]))
            except Exception:
                pass
    if len(boot_aucs) >= 10:
        ci = [float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))]
    else:
        ci = [auc, auc]

    logger.info(f"  {method_name}: AUC={auc:.3f} [{ci[0]:.3f},{ci[1]:.3f}], AP={auc_pr:.3f}")
    return {"auc": auc, "auc_pr": auc_pr, "prec_at_k": prec_at_k, "ci": ci}


def run_evaluation(scores_dict: Dict[str, np.ndarray], gt_info: Dict) -> Dict:
    """Evaluate all methods on suppression detection."""
    # Choose primary labels
    if gt_info["n_stacking"] >= 3:
        labels_primary = gt_info["labels_stacking"]
        label_name = "citation_stacking"
    elif gt_info["n_all"] >= 3:
        labels_primary = gt_info["labels_all"]
        label_name = "all_suppressions"
        logger.warning("Falling back to labels_all (too few stacking labels)")
    else:
        logger.warning("Too few positive labels for meaningful evaluation")
        labels_primary = gt_info["labels_all"]
        label_name = "all_suppressions"

    results = {}
    for method, scores in scores_dict.items():
        results[method] = evaluate_method(scores, labels_primary, method)

    # Key comparison: hodge_curl_z vs cidre
    if "hodge_curl_z" in scores_dict and "cidre" in scores_dict:
        if labels_primary.sum() >= 2:
            n_perm = 500  # reduced for demo speed
            rng = np.random.RandomState(42)
            s1 = np.where(np.isfinite(scores_dict["hodge_curl_z"]), scores_dict["hodge_curl_z"], 0.0)
            s2 = np.where(np.isfinite(scores_dict["cidre"]), scores_dict["cidre"], 0.0)
            try:
                delta_obs = roc_auc_score(labels_primary, s1) - roc_auc_score(labels_primary, s2)
                perm_deltas = []
                for _ in range(n_perm):
                    perm = rng.permutation(len(labels_primary))
                    try:
                        d = roc_auc_score(labels_primary[perm], s1) - roc_auc_score(labels_primary[perm], s2)
                        perm_deltas.append(d)
                    except Exception:
                        pass
                p_comp = float(np.mean(np.abs(perm_deltas) >= abs(delta_obs))) if perm_deltas else 1.0
                logger.info(f"Hodge-curl vs CIDRE: Δ={delta_obs:.3f}, p={p_comp:.4f}")
            except Exception as e:
                logger.warning(f"Comparison test failed: {e}")
                delta_obs, p_comp = 0.0, 1.0
        else:
            delta_obs, p_comp = 0.0, 1.0
    else:
        delta_obs, p_comp = 0.0, 1.0

    return {
        "label_primary": label_name,
        "n_positives_stacking": gt_info["n_stacking"],
        "n_positives_all": gt_info["n_all"],
        "methods": results,
        "curl_vs_cidre_delta_auc": float(delta_obs),
        "p_value_comparison": float(p_comp),
    }


scores_dict = {
    "hodge_curl_raw": hodge["node_curl_score"],
    "hodge_curl_z": z_info["z_score"],
    "hodge_curl_norm": degree_normalize_curl(hodge["node_curl_score"], C_active),
    "hodge_grad_residual": hodge["node_grad_residual"],
    **baselines,
}

gt_info = match_ground_truth(journals_raw, ground_truth, active_nodes)
suppressed_node_set = {
    int(active_idx) for active_idx, raw_idx in enumerate(active_nodes)
    if journals_raw[raw_idx].get("issn_l") in gt_issns
}

eval_results = run_evaluation(scores_dict, gt_info)

## Phase 7: Synthetic Cartel Injection Sensitivity Test

To assess detection sensitivity, we inject synthetic cartels (cyclic and reciprocal rings) into the real network and measure AUC as a function of cartel size k and citation weight factor w. This shows at what signal strength the Hodge curl detector fires.

In [ ]:
# ============================================================
# PHASE 7: SYNTHETIC CARTEL INJECTION
# ============================================================
def inject_cyclic_cartel(C: sparse.csr_matrix, k: int, w: float,
                          exclude: set, rng: np.random.RandomState) -> Tuple[sparse.csr_matrix, List[int]]:
    N = C.shape[0]
    available = [i for i in range(N) if i not in exclude]
    if len(available) < k:
        return C, []
    nodes = rng.choice(available, size=k, replace=False).tolist()
    C_mod = C.tolil()
    for idx in range(k):
        u, v = nodes[idx], nodes[(idx + 1) % k]
        C_mod[u, v] += w
    return C_mod.tocsr(), nodes


def inject_reciprocal_cartel(C: sparse.csr_matrix, k: int, w: float,
                               exclude: set, rng: np.random.RandomState) -> Tuple[sparse.csr_matrix, List[int]]:
    N = C.shape[0]
    available = [i for i in range(N) if i not in exclude]
    if len(available) < k:
        return C, []
    nodes = rng.choice(available, size=k, replace=False).tolist()
    C_mod = C.tolil()
    for u in nodes:
        for v in nodes:
            if u != v:
                C_mod[u, v] += w
    return C_mod.tocsr(), nodes


def fast_node_curl(C_mod: sparse.csr_matrix, edges: List[Tuple], triangles: List[Tuple], N: int) -> np.ndarray:
    """Compute node curl quickly using direct triangle method."""
    Y = C_mod - C_mod.T
    Y_arr = Y.toarray()
    node_curl = np.zeros(N)
    tri_count = np.zeros(N, dtype=int)
    for (i, j, k) in triangles:
        tc = abs(Y_arr[i, j] + Y_arr[j, k] - Y_arr[i, k])
        node_curl[i] += tc; node_curl[j] += tc; node_curl[k] += tc
        tri_count[i] += 1; tri_count[j] += 1; tri_count[k] += 1
    return node_curl / (tri_count + 1e-10)


def run_synthetic_injection(C_active: sparse.csr_matrix, edges: List[Tuple],
                              triangles: List[Tuple], N: int,
                              suppressed_node_set: set,
                              baseline_scores: Dict[str, np.ndarray]) -> List[Dict]:
    """Inject synthetic cartels and measure detection AUC."""
    if len(triangles) == 0:
        logger.warning("No triangles, skipping synthetic injection")
        return []

    mean_edge = float(C_active.data.mean()) if len(C_active.data) > 0 else 10.0
    cartel_types = ["cyclic", "reciprocal"]
    n_repeats = N_REPEATS

    records = []
    rng = np.random.RandomState(99)

    for cartel_type in cartel_types:
        for k in K_VALUES:
            for w_factor in W_FACTORS:
                w = w_factor * mean_edge
                auc_curls, auc_densities, auc_cidres, auc_recips = [], [], [], []

                for rep in range(n_repeats):
                    try:
                        if cartel_type == "cyclic":
                            C_mod, injected = inject_cyclic_cartel(C_active, k, w, suppressed_node_set, rng)
                        else:
                            C_mod, injected = inject_reciprocal_cartel(C_active, k, w, suppressed_node_set, rng)

                        if len(injected) < k:
                            continue

                        labels_inj = np.zeros(N)
                        for n in injected:
                            labels_inj[n] = 1

                        if labels_inj.sum() < 2:
                            continue

                        curl_mod = fast_node_curl(C_mod, edges, triangles, N)

                        # Updated reciprocity (fast)
                        recip_mod = compute_reciprocity(C_mod)

                        # Use pre-computed density (approximation for speed)
                        density_mod = baseline_scores["within_group_density"]
                        cidre_mod = baseline_scores["cidre"]

                        try:
                            auc_curls.append(roc_auc_score(labels_inj, curl_mod))
                            auc_densities.append(roc_auc_score(labels_inj, density_mod))
                            auc_cidres.append(roc_auc_score(labels_inj, cidre_mod))
                            auc_recips.append(roc_auc_score(labels_inj, recip_mod))
                        except Exception:
                            pass

                    except Exception as e:
                        logger.debug(f"Injection error: {e}")

                if auc_curls:
                    records.append({
                        "cartel_type": cartel_type,
                        "k": k,
                        "w_factor": w_factor,
                        "n_repeats": len(auc_curls),
                        "auc_hodge_curl_mean": float(np.mean(auc_curls)),
                        "auc_hodge_curl_std": float(np.std(auc_curls)),
                        "auc_cidre_mean": float(np.mean(auc_cidres)) if auc_cidres else None,
                        "auc_density_mean": float(np.mean(auc_densities)) if auc_densities else None,
                        "auc_reciprocity_mean": float(np.mean(auc_recips)) if auc_recips else None,
                    })

        logger.info(f"  Injection: {cartel_type} done")

    logger.info(f"Synthetic injection: {len(records)} condition records")
    return records


injection_records = run_synthetic_injection(
    C_active, edges, triangles, N, suppressed_node_set, baselines
)

## Phase 8: Confound Test

A key concern: might high curl simply reflect dense communities (which tend to have many internal citations)? The confound test computes the **partial correlation** of the curl z-score with the suppression label *after regressing out* citation density and reciprocity. A significant partial correlation confirms curl carries independent signal.

In [ ]:
# ============================================================
# PHASE 8: CONFOUND TEST
# ============================================================
def run_confound_test(C_active: sparse.csr_matrix, triangles: List[Tuple],
                       triangle_curls: np.ndarray, N: int,
                       comm_labels: np.ndarray, suppressed_node_set: set,
                       reciprocity_scores: np.ndarray, density_scores: np.ndarray,
                       z_scores: np.ndarray, labels_all: np.ndarray) -> Dict:
    """Test curl vs density separation between legitimate clusters and cartels."""
    from scipy.stats import mannwhitneyu

    # Identify dense legitimate communities (no suppressed journals)
    communities_dict = defaultdict(list)
    for node, c in enumerate(comm_labels):
        communities_dict[c].append(node)

    legitimate_comms = [
        members for members in communities_dict.values()
        if len(members) >= 5
        and not any(node in suppressed_node_set for node in members)
    ]

    # Compute group metrics
    def group_metrics(group_nodes: List[int]) -> Optional[Dict]:
        if len(group_nodes) < 2:
            return None
        g_set = set(group_nodes)
        sub = C_active[group_nodes][:, group_nodes]
        density = float(sub.sum()) / (len(group_nodes) * (len(group_nodes) - 1) + 1e-10)

        # Internal triangle curls
        if len(triangle_curls) > 0:
            internal_curl = sum(
                abs(float(triangle_curls[t_idx]))
                for t_idx, (i, j, k) in enumerate(triangles)
                if i in g_set and j in g_set and k in g_set
                and t_idx < len(triangle_curls)
            )
            n_tri = sum(
                1 for (i, j, k) in triangles
                if i in g_set and j in g_set and k in g_set
            )
        else:
            internal_curl = 0.0
            n_tri = 0

        curl_per_tri = internal_curl / (n_tri + 1e-10)
        mean_recip = float(reciprocity_scores[group_nodes].mean())

        return {
            "density": density,
            "curl_per_triangle": curl_per_tri,
            "internal_curl_total": internal_curl,
            "n_triangles": n_tri,
            "n_members": len(group_nodes),
            "mean_reciprocity": mean_recip,
        }

    # Sort by density, take top 5 legitimate
    legit_metrics = []
    for members in sorted(legitimate_comms,
                           key=lambda g: float(C_active[g][:, g].sum()) / (len(g) * (len(g) - 1) + 1e-10),
                           reverse=True)[:8]:
        m = group_metrics(members)
        if m:
            legit_metrics.append(m)

    # Cartel groups: suppressed journals near each other in the graph
    cartel_metrics = []
    if suppressed_node_set:
        # Group suppressed nodes by community
        supp_comms = defaultdict(list)
        for node in suppressed_node_set:
            if node < N:
                supp_comms[comm_labels[node]].append(node)
        for members in supp_comms.values():
            if len(members) >= 2:
                m = group_metrics(members)
                if m:
                    cartel_metrics.append(m)

    # Statistical tests
    results = {
        "legit_clusters": legit_metrics,
        "cartel_groups": cartel_metrics,
        "mannwhitney_density_p": None,
        "mannwhitney_curl_p": None,
        "partial_corr_curl": None,
        "partial_corr_ci": [None, None],
    }

    if legit_metrics and cartel_metrics:
        try:
            legit_densities = [m["density"] for m in legit_metrics]
            cartel_densities = [m["density"] for m in cartel_metrics]
            legit_curls = [m["curl_per_triangle"] for m in legit_metrics]
            cartel_curls = [m["curl_per_triangle"] for m in cartel_metrics]

            if len(legit_densities) >= 2 and len(cartel_densities) >= 2:
                _, p_density = mannwhitneyu(legit_densities, cartel_densities, alternative="two-sided")
                results["mannwhitney_density_p"] = float(p_density)
            if len(legit_curls) >= 2 and len(cartel_curls) >= 2:
                _, p_curl = mannwhitneyu(legit_curls, cartel_curls, alternative="two-sided")
                results["mannwhitney_curl_p"] = float(p_curl)
        except Exception as e:
            logger.warning(f"Mann-Whitney test failed: {e}")

    # Partial correlation: curl after regressing out density + reciprocity
    try:
        from sklearn.linear_model import LinearRegression
        finite_mask = np.isfinite(z_scores) & np.isfinite(density_scores) & np.isfinite(reciprocity_scores)
        if finite_mask.sum() >= 20:
            X = np.stack([density_scores[finite_mask], reciprocity_scores[finite_mask]], axis=1)
            y_label = labels_all[finite_mask]
            y_curl = z_scores[finite_mask]

            model_label = LinearRegression().fit(X, y_label)
            model_curl = LinearRegression().fit(X, y_curl)
            label_resid = y_label - model_label.predict(X)
            curl_resid = y_curl - model_curl.predict(X)

            if label_resid.std() > 1e-10 and curl_resid.std() > 1e-10:
                partial_r = float(np.corrcoef(curl_resid, label_resid)[0, 1])
                rng = np.random.RandomState(42)
                boot_rs = []
                for _ in range(CONFOUND_BOOT):
                    idx = rng.randint(0, len(label_resid), len(label_resid))
                    if label_resid[idx].std() > 1e-10:
                        boot_rs.append(float(np.corrcoef(curl_resid[idx], label_resid[idx])[0, 1]))
                if boot_rs:
                    results["partial_corr_curl"] = partial_r
                    results["partial_corr_ci"] = [
                        float(np.percentile(boot_rs, 2.5)),
                        float(np.percentile(boot_rs, 97.5)),
                    ]
                    logger.info(f"Partial correlation (curl|density,recip): r={partial_r:.3f}")
    except Exception as e:
        logger.warning(f"Partial correlation failed: {e}")

    return results


confound = run_confound_test(
    C_active, triangles, hodge["triangle_curls"], N,
    comm_labels, suppressed_node_set,
    baselines["reciprocity"], baselines["within_group_density"],
    z_info["z_score"], gt_info["labels_all"]
)

## Results & Visualization

Summary of all key metrics from this demo run, plus plots showing:
1. Hodge energy decomposition (grad / curl / harm fractions)
2. AUC comparison across detection methods
3. Curl z-score distribution: cartel vs non-cartel nodes
4. Pre-computed scores from `mini_demo_data.json` (full-scale run)

In [ ]:
# ============================================================
# RESULTS & VISUALIZATION
# ============================================================
print("=" * 60)
print("DEMO RUN RESULTS")
print("=" * 60)
print(f"Network: N={N} journals, E={E} edges, T={T} triangles")
print(f"Cartel nodes: {len(suppressed_node_set)} / {N}")
print()
print("Hodge energy decomposition:")
print(f"  Gradient (prestige):  {hodge['grad_fraction']:.3f}")
print(f"  Curl (cyclic flows):  {hodge['curl_fraction']:.3f}")
print(f"  Harmonic (residual):  {hodge['harm_fraction']:.3f}")
print()
print("Detection AUC-ROC:")
methods_order = [
    ('hodge_grad_residual', 'Hodge Grad Residual'),
    ('hodge_curl_raw',     'Hodge Curl Raw'),
    ('hodge_curl_norm',    'Hodge Curl Norm'),
    ('hodge_curl_z',       'Hodge Curl Z'),
    ('cidre',              'CIDRE (baseline)'),
    ('reciprocity',        'Reciprocity'),
    ('within_group_density','Within-Group Density'),
    ('pagerank',           'PageRank'),
]
eval_methods = eval_results.get('methods', {})
for key, label in methods_order:
    res = eval_methods.get(key, {})
    auc = res.get('auc')
    ci = res.get('ci', [None, None])
    if auc is not None:
        print(f"  {label:<30}: AUC={auc:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")
    else:
        print(f"  {label:<30}: N/A (too few positives)")

print()
print(f"Confound: partial_corr(curl|density,recip) = {confound.get('partial_corr_curl')}")
if injection_records:
    print(f"\nInjection sensitivity: {len(injection_records)} conditions")
    for r in injection_records:
        print(f"  {r['cartel_type']:10} k={r['k']} w={r['w_factor']:.2f}: "
              f"AUC_Hodge={r['auc_hodge_curl_mean']:.3f} vs CIDRE={r.get('auc_cidre_mean', 'N/A')}")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Hodge energy pie chart
ax = axes[0]
fracs = [hodge['grad_fraction'], hodge['curl_fraction'], hodge['harm_fraction']]
labels_pie = ['Gradient\n(prestige)', 'Curl\n(cyclic)', 'Harmonic\n(residual)']
colors = ['#4CAF50', '#F44336', '#2196F3']
wedges, texts, autotexts = ax.pie(fracs, labels=labels_pie, colors=colors,
                                   autopct='%1.1f%%', startangle=90)
ax.set_title('Hodge Energy Decomposition')

# Plot 2: AUC bar chart
ax = axes[1]
auc_values = []
auc_labels = []
auc_colors = []
hodge_keys = {'hodge_grad_residual', 'hodge_curl_raw', 'hodge_curl_norm', 'hodge_curl_z'}
for key, label in methods_order:
    res = eval_methods.get(key, {})
    auc = res.get('auc')
    if auc is not None:
        auc_values.append(auc)
        auc_labels.append(label.replace(' ', '\n'))
        auc_colors.append('#F44336' if key in hodge_keys else '#9E9E9E')
if auc_values:
    bars = ax.bar(range(len(auc_values)), auc_values, color=auc_colors)
    ax.set_xticks(range(len(auc_values)))
    ax.set_xticklabels(auc_labels, fontsize=7)
    ax.axhline(0.5, color='black', linestyle='--', alpha=0.5, label='random')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('AUC-ROC')
    ax.set_title('Detection AUC by Method\n(red=Hodge, grey=baseline)')
    ax.legend(fontsize=8)

# Plot 3: Curl z-score distribution — cartel vs non-cartel
ax = axes[2]
z = z_info['z_score']
labels_arr = gt_info['labels_stacking'] if gt_info['n_stacking'] > 0 else gt_info['labels_all']
cartel_mask = labels_arr == 1
legit_mask = labels_arr == 0
z_finite = np.where(np.isfinite(z), z, 0.0)
ax.hist(z_finite[legit_mask], bins=30, alpha=0.6, color='#4CAF50', label='Non-cartel', density=True)
if cartel_mask.sum() > 0:
    ax.hist(z_finite[cartel_mask], bins=10, alpha=0.8, color='#F44336', label='Cartel', density=True)
ax.set_xlabel('Curl z-score')
ax.set_ylabel('Density')
ax.set_title('Curl Z-score Distribution')
ax.legend()
ax.axvline(0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(str(WORKSPACE / 'results' / 'hodge_demo_results.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved to results/hodge_demo_results.png")

# --- Show pre-computed scores from full-scale run (mini_demo_data.json) ---
print()
print("=" * 60)
print("PRE-COMPUTED SCORES (mini_demo_data.json — full-scale run)")
print("=" * 60)
meta = data['metadata']
print(f"N={meta['n_journals']}, T={meta['n_triangles']}, N_NULL={meta['n_null_samples']}")
print(f"Hodge curl AUC: {meta['hodge_curl_auc_roc']:.3f}")
print(f"CIDRE AUC:      {meta['cidre_auc_roc']:.3f}")
print(f"Δ(Hodge-CIDRE): {meta['delta_auc_hodge_minus_cidre']:.3f}")
print(f"Hodge energy:   grad={meta['hodge_energy_fractions']['gradient']:.3f}, "
      f"curl={meta['hodge_energy_fractions']['curl']:.3f}, "
      f"harm={meta['hodge_energy_fractions']['harmonic']:.3f}")
print()
print("Top-5 by Hodge curl z-score (from pre-computed data):")
exs = data['datasets'][0]['examples']
exs_sorted = sorted(exs, key=lambda e: float(e['predict_hodge_curl_z']), reverse=True)
for ex in exs_sorted[:5]:
    print(f"  {ex['metadata_journal_name'][:35]:<35}  "
          f"curl_z={ex['predict_hodge_curl_z']}  label={ex['output']}")